# Dataset

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision.datasets import CIFAR10
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import os
import time

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset parameters
data_dir = './data'
labels = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = len(labels)
batch_size = 3200

# Data augmentation and normalization
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Load CIFAR10 dataset
train_dataset = CIFAR10(root='./data', train=True, download=True, transform=train_transform)
val_dataset = CIFAR10(root='./data', train=False, download=True, transform=val_transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


100%|██████████| 170M/170M [00:03<00:00, 43.0MB/s]


# Model

In [2]:

# Baseline CNN Model
class BaselineCNN(nn.Module):
    def __init__(self, num_classes):
        super(BaselineCNN, self).__init__()
        self.features = nn.Sequential(
            # First block: 32x32x3 -> 32x32x64
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 16x16x64

            # Second block: 16x16x64 -> 16x16x128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 8x8x128

            # Third block: 8x8x128 -> 8x8x256
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 4x4x256
        )

        # Classifier: 4*4*256 = 4096
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(4096, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

        # Initialize weights
        self._initialize_weights()

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)  # flatten all dimensions except batch
        x = self.classifier(x)
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)


# Baseline Training

In [3]:

# Training function with validation
def train_model(model, criterion, optimizer, num_epochs=5, log_dir=None):
    writer = SummaryWriter(log_dir=log_dir) if log_dir else None
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Training phase
        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(train_dataset)
        epoch_acc = running_corrects.double() / len(train_dataset)

        print(f'Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

        # Validation phase
        model.eval()
        running_loss = 0.0
        running_corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

        val_loss = running_loss / len(val_dataset)
        val_acc = running_corrects.double() / len(val_dataset)

        print(f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

        if writer:
            writer.add_scalars('Loss', {
                'train': epoch_loss,
                'val': val_loss
            }, epoch)
            writer.add_scalars('Accuracy', {
                'train': epoch_acc,
                'val': val_acc
            }, epoch)

        # Save best model
        if val_acc > best_acc:
            best_acc = val_acc
            if log_dir:
                torch.save(model.state_dict(), os.path.join(log_dir, 'best_model.pth'))

    return model

# Initialize and train baseline model
baseline_model = BaselineCNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(baseline_model.parameters(), lr=0.001, weight_decay=1e-4)

# Create logs directory if it doesn't exist
os.makedirs('./logs', exist_ok=True)

# Train the model
baseline_model = train_model(baseline_model, criterion, optimizer, num_epochs=1,
                           log_dir=f'./logs/baseline_{int(time.time())}')

Epoch 1/1
----------
Train Loss: 2.0078 Acc: 0.2550
Val Loss: 2.0196 Acc: 0.2655


# GPU Memory Clean

In [4]:
import gc
model_state = baseline_model.state_dict()

# Clear everything
del train_loader, val_loader, train_dataset, val_dataset
gc.collect()
torch.cuda.empty_cache()

# Model can be loaded later if needed:
baseline_model.load_state_dict(model_state)


<All keys matched successfully>

# Usando MobileNetV2 (https://arxiv.org/abs/1801.04381)

In [5]:
# Load CIFAR10 dataset
train_dataset = CIFAR10(root='./data', train=True, download=True, transform=train_transform)
val_dataset = CIFAR10(root='./data', train=False, download=True, transform=val_transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [6]:
# Transfer Learning with MobileNet
model_ft = models.mobilenet_v2(pretrained=True)

# Modify first convolution layer to handle 32x32 images
model_ft.features[0][0] = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False)

# Freeze all parameters except first conv and classifier
for param in model_ft.parameters():
    param.requires_grad = False

# Replace classifier
model_ft.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model_ft.last_channel, num_classes)
)

# Unfreeze first conv layer
for param in model_ft.features[0].parameters():
    param.requires_grad = True

model_ft = model_ft.to(device)

# Train classifier
optimizer_ft = optim.RMSprop([
    {'params': model_ft.features[0].parameters()},
    {'params': model_ft.classifier.parameters()}
], lr=0.001)

criterion = nn.CrossEntropyLoss()
model_ft = train_model(model_ft, criterion, optimizer_ft, num_epochs=1,
                      log_dir=f'./logs/mobilenet_trf_{int(time.time())}')

# Fine-tuning
# Unfreeze last few layers
for param in model_ft.features[-4:].parameters():
    param.requires_grad = True

optimizer_ft = optim.SGD([
    {'params': model_ft.features.parameters(), 'lr': 1e-4},
    {'params': model_ft.classifier.parameters(), 'lr': 1e-3}
], momentum=0.9)

model_ft = train_model(model_ft, criterion, optimizer_ft, num_epochs=1,
                      log_dir=f'./logs/mobilenet_finetune_{int(time.time())}')


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 98.2MB/s]

Epoch 1/1
----------


Train Loss: 1.8833 Acc: 0.3524
Val Loss: 1.7277 Acc: 0.4107
Epoch 1/1
----------
Train Loss: 1.5449 Acc: 0.4531
Val Loss: 1.4128 Acc: 0.5159
